In [5]:
# elasticsearch 서버에 인덱스들을 모두 출력하는 코드

from elasticsearch import Elasticsearch
import warnings

# 보안 경고 무시
warnings.filterwarnings('ignore')

# 1. Elasticsearch 접속 (환경에 맞게 주소를 수정하세요)
es = Elasticsearch("http://192.168.0.40:9200")

# 2. 모든 인덱스 정보 가져오기 (h="index,docs.count" 옵션으로 필요한 정보만 요청)
# s="index"는 이름을 기준으로 정렬합니다.
indices_stats = es.cat.indices(h="index,docs.count", format="json")

print(f"{'Index Name':<30} | {'Document Count':<15}")
print("-" * 50)

found_any = False
for item in indices_stats:
    index_name = item['index']
    
    # .으로 시작하는 시스템 인덱스는 건너뛰기
    if index_name.startswith('.'):
        continue
    
    doc_count = item['docs.count']
    # 데이터가 없을 경우 'None'이나 공백으로 올 수 있어 0으로 처리
    doc_count = int(doc_count) if doc_count and doc_count != 'None' else 0
    
    print(f"{index_name:<30} | {doc_count:<15,}")
    found_any = True

if not found_any:
    print("일반 인덱스가 존재하지 않습니다.")

Index Name                     | Document Count 
--------------------------------------------------
qaindex_128_10_20241115        | 600            
prequery_128                   | 124            
ezisc_qa_128_10_20241218       | 991            
ezisc_qa_128_10_20260107       | 1,031          
company_128_10_20241115        | 600            


In [4]:
# 특정인덱스를 삭제하는 코드

from elasticsearch import Elasticsearch
import warnings

# 보안 경고 무시
warnings.filterwarnings('ignore')

# 1. Elasticsearch 접속
es = Elasticsearch("http://192.168.0.40:9200")

# 2. 삭제할 인덱스명 설정
target_index = "ezisc_qa_128_10_20241115"

def delete_specific_index(index_name):
    try:
        # 인덱스가 존재하는지 먼저 확인
        if es.indices.exists(index=index_name):
            # 인덱스 삭제 실행
            response = es.indices.delete(index=index_name)
            
            if response.get('acknowledged'):
                print(f"✅ 성공: '{index_name}' 인덱스가 정상적으로 삭제되었습니다.")
            else:
                print(f"⚠️ 경고: '{index_name}' 삭제 요청은 보냈으나 확인되지 않았습니다.")
        else:
            print(f"ℹ️ 알림: '{index_name}' 인덱스가 존재하지 않아 삭제할 필요가 없습니다.")
            
    except Exception as e:
        print(f"❌ 오류 발생: {e}")

# 함수 실행
delete_specific_index(target_index)

✅ 성공: 'ezisc_qa_128_10_20241115' 인덱스가 정상적으로 삭제되었습니다.


In [2]:
# 인덱스명을 입력하면 해당 인덱스에 1,2번째 저장된 데이터들을 출력하는 코드 

from elasticsearch import Elasticsearch
import json
import warnings

# 보안 경고 무시
warnings.filterwarnings('ignore')

# 1. Elasticsearch 접속
es = Elasticsearch("http://192.168.0.40:9200")

# 2. 조회할 인덱스명 설정 (원하는 인덱스명으로 수정하세요)
target_index = "ezisc_qa_128_10_20260107"

def get_first_two_docs(index_name):
    try:
        # 인덱스 존재 여부 확인
        if not es.indices.exists(index=index_name):
            print(f"❌ '{index_name}' 인덱스가 존재하지 않습니다.")
            return

        # 상위 2개의 문서 검색 (size=2)
        # 별도의 query가 없으면 match_all로 동작합니다.
        response = es.search(index=index_name, size=2)
        
        hits = response['hits']['hits']
        count = len(hits)
        
        if count == 0:
            print(f"Empty: '{index_name}' 인덱스에 저장된 데이터가 없습니다.")
        else:
            print(f"🔍 '{index_name}' 인덱스에서 총 {count}개의 문서를 찾았습니다.\n")
            
            for i, hit in enumerate(hits, 1):
                print(f"--- [{i}번째 문서 상세 내용 (ID: {hit['_id']})] ---")
                # JSON 형태로 예쁘게 출력 (indent=4)
                print(json.dumps(hit['_source'], indent=4, ensure_ascii=False))
                print("\n")

    except Exception as e:
        print(f"오류 발생: {e}")

# 함수 실행
get_first_two_docs(target_index)

🔍 'ezisc_qa_128_10_20260107' 인덱스에서 총 2개의 문서를 찾았습니다.

--- [1번째 문서 상세 내용 (ID: F0MO25wByt_wHjPerwn1)] ---
{
    "rfile_name": "1_1",
    "rfile_text": "[제이피에스][받은메일][20191231] EZis-C 로그인 실패 에러메시지\n\nQ:\n한 PC에서 로그인 실패 난다. 일단 PWD를 초기화 하니깐..잘되는데, 한 5일전부터 이런 증상이 발생 했다고 한다. 로그 파악해달라.\n\nA:\n로그를 보니깐..로그인 실패시 SC_LOGIN_LOCK_UID:  211: 계정 잠김(암호 5회이상 오류)오류가 발생하고 있다.\n211 에러는 3가지중 하나인데...암호 5회이상 오류인 경우, 사용기간 만료인 경우, 비밀번호 정책에서 로그인 잠금(며칠동안 미접속시...) 등이 있다.",
    "vector1": [
        -0.11199951171875,
        0.029876708984375,
        0.048248291015625,
        0.021270751953125,
        0.013153076171875,
        0.0184783935546875,
        0.037567138671875,
        0.1817626953125,
        -0.035003662109375,
        0.06292724609375,
        0.046051025390625,
        -0.032073974609375,
        0.010162353515625,
        0.0198516845703125,
        0.09423828125,
        0.01297760009765625,
        0.0716552734375,
        0.06719970703125,
        -0.0625,
        0.02435302734375,
        0.10

In [ ]:
es

In [ ]:
import elasticsearch
print(elasticsearch.__version__)

In [ ]:
stats = es.indices.stats()

In [ ]:
indices_info = stats['indices']
print(f"{'Index Name':<30} | {'Document Count':<15}")
print("-" * 60)